In [ ]:
from atlite.gis import ExclusionContainer, shape_availability, padded_transform_and_shape
import time
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio as rio
from rasterio.enums import Resampling
from rasterio.features import rasterize
from numpy import empty
from scipy.ndimage import binary_dilation as dilation
from rasterio.features import geometry_mask

from workflow.scripts.extra_functions import get_bounding

In [ ]:
country_codes = {
    "Austria" : "AT",
    "Belgium" : "BE",
    "Bulgaria" : "BG", #
    "Switzerland" : "CH", #
    "Czech Republic" : "CZ", #
    "Germany" : "DE",
    "Denmark" : "DK",
    "Estonia" : "EE" ,
    "Spain" : "ES",
    "Finland" : "FI", #
    "France" : "FR",
    "United Kingdom" : "UK",
    "Greece" : "GR",
    "Croatia" : "HR", #
    "Hungary" : "HU", #
    "Ireland" : "IE",
    "Italy" : "IT",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Latvia" : "LV",
    "Netherlands" : "NL",
    "Norway" : "NO",
    "Poland" : "PL", #
    "Portugal" : "PT",
    "Romania" : "RO", #
    "Sweden" : "SE",
    "Slovenia" : "SI", #
    "Slovakia" : "SK" #
}


In [ ]:
## method
method = snakemake.params.method # atlite

## input data
scenario_path = snakemake.input.scenario_path
europe_onshore_shp = snakemake.input.europe_onshore_shapefile
scenario = snakemake.wildcards.scenario

# params
polygon_borders = snakemake.params.polygon_borders
input_scenarios = snakemake.params.input_scenarios["LandscapeVisualImpact"]
target_crs = snakemake.params.target_crs # target EPSG

## output data
output_path = snakemake.output.landscape_visual_impact_path


In [ ]:
# resolution
res = 100

# borders
[rectx1, rectx2, recty1, recty2] = polygon_borders
polygon_bounds = get_bounding(target_crs,rectx1,rectx2,recty1,recty2)

# codes
codes = input_scenarios[scenario]

In [ ]:
europe = (
    gpd
    .read_file(europe_onshore_shp)
    .set_index(["index"])
    .loc[:,['geometry']]
    .to_crs(target_crs)
)

onshore = (
    europe
    .assign(area = 1)
    .dissolve(by='area')
    .reset_index()
)
onshore["area"] = onshore["area"].astype("uint8")

In [ ]:
country_no_excluded = snakemake.params.country_no_excluded
t=time.time()

if method=="manual":
    dst_transform, dst_shape = padded_transform_and_shape(
        bounds=rio.features.bounds(polygon_bounds), # cutout bounds
        res=res # resolution
    )

    src = rio.open(scenario_path)
    raster_scenicness = src.read(1) 
    raster_scenicness_transform = src.transform  # Get the raster's transform

    raster_reprojected = empty(dst_shape)
    
    rio.warp.reproject(raster_scenicness,
            raster_reprojected,
            src_transform=raster_scenicness_transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=target_crs,
            resampling=Resampling.nearest,
            num_threads=2)

    if isinstance(codes, list): # it is a list
        # Create a raster based on the list of codes
        raster_reprojected = np.where(np.isin(raster_reprojected,codes), 1, 0)
        # put areas back for some countries
        country_removed = country_no_excluded[snakemake.wildcards.scenario]
        for country_name in country_removed:
            country_geom = europe.query("index==@country_name").geometry

            country_mask = ~geometry_mask(
                country_geom,
                out_shape=dst_shape,
                transform=dst_transform,
                all_touched=True,
                )

            raster_reprojected[country_mask] = 0
        
        # raster europe geometries
        europe_mask = ~geometry_mask(
            onshore.geometry,
            out_shape=dst_shape,
            transform=dst_transform,
            all_touched=True,
            )

        raster_reprojected = europe_mask & raster_reprojected.astype(bool)
        
        raster_countries_scenicness = raster_reprojected


    else:
        df = pd.read_csv(codes)
        df["country"] = df["country"].replace(country_codes)
        df = df.set_index("country")
        df = df.replace({0:-1}) # replace 0 values with any other
        df["exclude_scenic"] = df["exclude_scenic"]*1
        df["exclude_highly_scenic"] = df["exclude_highly_scenic"]*2
        countries_codes = df

        raster_countries_scenicness = np.zeros(dst_shape, dtype=np.uint8)

        # Create a loop through each country to buffer settlements
        for country, country_exclusions_codes in countries_codes.iterrows():
            country_codes = list(country_exclusions_codes)
            country_name = country
            country_geom = europe.query("index==@country").geometry
            
            country_mask = ~geometry_mask(
                country_geom,
                out_shape=dst_shape,
                transform=dst_transform,
                all_touched=True,
                )
            
            ### Country specific values
            # Create a boolean mask containing landscape scenicness
            # containing the specific code
            # True: code is in the pixel
            # False: code is not in the pixel

            scenicness_mask = (np.where(np.isin(raster_reprojected,country_codes), 1, 0)).astype(bool)

            # select pixels within a country and correspond to the scenicness excluded
            country_scenicness_mask = scenicness_mask & country_mask
            
            # Add the excluded areas to the array
            raster_countries_scenicness[country_scenicness_mask] = 1

    raster_countries_scenicness = raster_countries_scenicness.astype("uint8")

    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_countries_scenicness.shape[1],
        'height': raster_countries_scenicness.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': dst_transform,
        'compress': 'lzw'
    }

    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_countries_scenicness, 1)

    print(time.time()-t)

medium
33.304224491119385
high
63.46202349662781


In [ ]:
country_no_excluded = snakemake.params.country_no_excluded

t=time.time()
if method=="atlite":
    
    exclusion_container = ExclusionContainer(crs=target_crs)
    exclusion_container.add_raster(scenario_path, codes=codes, crs=target_crs)
    # exclusion_container.add_geometry(onshore.geometry, invert=True)
    masked, transform = shape_availability(polygon_bounds.geometry, exclusion_container)
    raster_reprojected = (~masked)

    exclusion_container = ExclusionContainer(crs=target_crs)
    exclusion_container.add_geometry(onshore.geometry)
    masked, _ = shape_availability(polygon_bounds.geometry, exclusion_container)
    raster_onshore = (~masked)

    raster_reprojected = (raster_reprojected & raster_onshore).astype("uint8")

    
    country_removed = country_no_excluded[snakemake.wildcards.scenario]
    for country_name in country_removed:
        country_geom = europe.query("index==@country_name").geometry

        country_mask = ~geometry_mask(
            country_geom,
            out_shape=dst_shape,
            transform=dst_transform,
            all_touched=True,
            )

        raster_reprojected[country_mask] = 0
    raster_reprojected = raster_reprojected.astype("uint8")

    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_reprojected.shape[1],
        'height': raster_reprojected.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': transform,
        'compress': 'lzw'
    }

    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_reprojected, 1)

    print(time.time()-t)

In [ ]:
if method=="test":
    f = open(output_path, "x")